# Audio Popularity Classification

## 1. Setup

In [ ]:
%pip install -q lightgbm==4.6.0 xgboost==3.2.0 catboost==1.2.10 scikit-learn==1.7.2

import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedGroupKFold

SEEDS = [0, 1, 2, 3, 4]
N_FOLDS = 10
MODEL_NAMES = ['lightgbm_tuned', 'xgboost_tuned', 'catboost_final', 'extra_trees']
TASK_TYPE = 'CPU'

In [2]:
TARGET = "Popularity_Type"
AUDIO_FEATURES = ["speechiness", "danceability", "duration_ms", "energy", "instrumentalness", "valence", "tempo",
         "loudness", "acousticness", "liveness"]
CATEGORICAL_FEATURES = ["key", "mode", "time_signature"]
KNN_AUDIO_FEATURES = ["speechiness", "danceability", "energy", "instrumentalness", "valence", "loudness", "acousticness",
             "liveness", "tempo", "duration_ms"]

In [ ]:
required_files = ['train.csv', 'test.csv', 'sample_submission.csv']
missing_files = [name for name in required_files if not os.path.exists(name)]
if missing_files:
    print('Upload train.csv, test.csv, and sample_submission.csv')
    files.upload()

train_data = pd.read_csv('train.csv').dropna(subset=['uri']).reset_index(drop=True)
test_data = pd.read_csv('test.csv')
sample_submission = pd.read_csv('sample_submission.csv')
train_labels = (train_data[TARGET] == 'High').astype(int).values
train_data['high_popularity_label'] = train_labels
print(f'Train: {len(train_data):,} rows; test: {len(test_data):,} rows; High share: {train_labels.mean():.1%}')
display(train_data[TARGET].value_counts().rename('rows').to_frame())

In [4]:
def build_release_date_features(track_data):
    release_date_strings = track_data["track_album_release_date"].astype(str)
    feature_values = pd.DataFrame(index=track_data.index)
    feature_values["release_year"] = pd.to_numeric(release_date_strings.str[:4], errors="coerce")
    feature_values["release_month"] = pd.to_numeric(release_date_strings.str.extract(r"^\d{4}-(\d{2})", expand=False), errors="coerce")
    feature_values["release_day"] = pd.to_numeric(release_date_strings.str.extract(r"^\d{4}-\d{2}-(\d{2})", expand=False), errors="coerce")
    date_precision = np.select([release_date_strings.str.match(r"^\d{4}$"), release_date_strings.str.match(r"^\d{4}-\d{2}$")], [0, 1], default=2)
    feature_values["release_date_precision"] = date_precision
    exact_release_dates = pd.to_datetime(release_date_strings.where(date_precision == 2), format="%Y-%m-%d", errors="coerce")
    month_release_dates = pd.to_datetime(release_date_strings.where(date_precision == 1) + "-15", format="%Y-%m-%d", errors="coerce")
    year_release_dates = pd.to_datetime(release_date_strings.where(date_precision == 0) + "-07-01", format="%Y-%m-%d", errors="coerce")
    filled_release_dates = exact_release_dates.fillna(month_release_dates).fillna(year_release_dates)
    feature_values["release_days_since_1950"] = (filled_release_dates - pd.Timestamp("1950-01-01")).dt.days
    feature_values["track_age_days"] = (pd.Timestamp("2024-12-31") - filled_release_dates).dt.days
    feature_values["log_track_age_days"] = np.log1p(feature_values["track_age_days"].clip(lower=0))
    feature_values["release_day_of_week"] = exact_release_dates.dt.dayofweek
    feature_values["released_on_friday"] = (feature_values["release_day_of_week"] == 4).astype(float).where(feature_values["release_day_of_week"].notna())
    feature_values["release_day_of_year"] = exact_release_dates.dt.dayofyear
    return feature_values

In [5]:
BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True})

## 2. Feature engineering

In [ ]:
def build_audio_interaction_features(track_data):
    feature_values = pd.DataFrame(index=track_data.index)
    feature_values["log_duration_milliseconds"] = np.log1p(track_data["duration_ms"].clip(lower=0))
    feature_values["loudness_times_energy"] = track_data["loudness"] * track_data["energy"]
    feature_values["energy_minus_acousticness"] = track_data["energy"] - track_data["acousticness"]
    feature_values["danceability_times_valence"] = track_data["danceability"] * track_data["valence"]
    feature_values["danceability_times_energy"] = track_data["danceability"] * track_data["energy"]
    feature_values["log_instrumentalness"] = np.log10(track_data["instrumentalness"] + 1e-6)
    feature_values["log_speechiness"] = np.log10(track_data["speechiness"] + 1e-4)
    feature_values["log_acousticness"] = np.log10(track_data["acousticness"] + 1e-6)
    feature_values["log_liveness"] = np.log10(track_data["liveness"] + 1e-4)
    feature_values["tempo_distance_from_120"] = np.abs(track_data["tempo"] - 120)
    feature_values["loudness_per_energy"] = track_data["loudness"] / (track_data["energy"] + 1e-3)
    return feature_values

def build_static_features(train_data, test_data):
    train_feature_parts = [train_data[AUDIO_FEATURES + CATEGORICAL_FEATURES].copy(), build_audio_interaction_features(train_data), build_release_date_features(train_data)]
    test_feature_parts = [test_data[AUDIO_FEATURES + CATEGORICAL_FEATURES].copy(), build_audio_interaction_features(test_data), build_release_date_features(test_data)]

    train_feature_parts.append(train_data.groupby("uri")["uri"].transform("count").rename("track_occurrence_count").to_frame())
    test_feature_parts.append(test_data.groupby("uri")["uri"].transform("count").rename("track_occurrence_count").to_frame())

    combined_unique_tracks = pd.concat([train_data[["uri", "track_album_release_date"]], test_data[["uri", "track_album_release_date"]]]).drop_duplicates("uri")
    date_group_counts = combined_unique_tracks.groupby("track_album_release_date")["uri"].count()
    train_feature_parts.append(train_data["track_album_release_date"].map(date_group_counts).rename("same_release_date_track_count").to_frame())
    test_feature_parts.append(test_data["track_album_release_date"].map(date_group_counts).rename("same_release_date_track_count").to_frame())
    return pd.concat(train_feature_parts, axis=1), pd.concat(test_feature_parts, axis=1)

In [ ]:
def compute_smoothed_target_encodings(fit_keys, reference_labels, apply_keys, prior_weight=3.0):
    fit_keys = pd.Series(fit_keys).astype(str).values; apply_keys = pd.Series(apply_keys).astype(str).values
    global_high_rate = float(np.mean(reference_labels))
    group_label_statistics = pd.DataFrame({"category_key": fit_keys, "high_popularity_label": reference_labels}).groupby("category_key")["high_popularity_label"].agg(["sum", "count"])
    smoothed_label_means = (group_label_statistics["sum"] + prior_weight * global_high_rate) / (group_label_statistics["count"] + prior_weight)
    return np.column_stack([pd.Series(apply_keys).map(smoothed_label_means).fillna(global_high_rate).values,
                            pd.Series(apply_keys).map(group_label_statistics["count"]).fillna(0).values])

def compute_out_of_fold_target_encodings(keys, labels, groups, seed, prior_weight=3.0):
    feature_values = np.zeros((len(labels), 2))
    inner_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for inner_train_indices, inner_validation_indices in inner_splitter.split(keys, labels, groups=groups):
        feature_values[inner_validation_indices] = compute_smoothed_target_encodings(keys[inner_train_indices], labels[inner_train_indices], keys[inner_validation_indices], prior_weight)
    return feature_values

def compute_same_date_neighbor_features(reference_audio_features, reference_release_dates, reference_labels, query_audio_features, query_release_dates, exclude_self=False, neighbor_count=5, prior=2.0):
    global_high_rate = reference_labels.mean()
    neighbor_label_means = np.full(len(query_audio_features), global_high_rate); nearest_neighbor_distances = np.full(len(query_audio_features), np.nan)
    indices_by_release_date = {}
    for reference_index, release_date in enumerate(reference_release_dates):
        indices_by_release_date.setdefault(release_date, []).append(reference_index)
    for query_index, release_date in enumerate(query_release_dates):
        neighbor_indices = indices_by_release_date.get(release_date)
        if not neighbor_indices:
            continue
        neighbor_indices = np.array(neighbor_indices)
        distances = np.sqrt(((reference_audio_features[neighbor_indices] - query_audio_features[query_index]) ** 2).sum(1))
        if exclude_self:
            nonidentical_neighbor_mask = distances > 1e-9
            neighbor_indices, distances = neighbor_indices[nonidentical_neighbor_mask], distances[nonidentical_neighbor_mask]
            if len(neighbor_indices) == 0:
                continue
        nearest_neighbor_order = np.argsort(distances)[:neighbor_count]
        neighbor_weights = 1.0 / (1.0 + distances[nearest_neighbor_order])
        neighbor_label_means[query_index] = (np.sum(neighbor_weights * reference_labels[neighbor_indices[nearest_neighbor_order]]) + prior * global_high_rate) / (np.sum(neighbor_weights) + prior)
        nearest_neighbor_distances[query_index] = distances[nearest_neighbor_order][0]
    return neighbor_label_means, nearest_neighbor_distances

In [ ]:
def build_fold_feature_matrices(train_features, test_features, train_data, test_data, train_labels, folds, seed):
    release_dates = train_data["track_album_release_date"].astype(str).values; test_release_dates = test_data["track_album_release_date"].astype(str).values
    track_uris = train_data["uri"].values
    combined_audio_features = pd.concat([train_data[KNN_AUDIO_FEATURES], test_data[KNN_AUDIO_FEATURES]])
    combined_audio_features = combined_audio_features.assign(duration_ms=np.log1p(combined_audio_features["duration_ms"]), instrumentalness=np.log10(combined_audio_features["instrumentalness"] + 1e-6))
    audio_feature_means, audio_feature_stds = combined_audio_features.mean(), combined_audio_features.std()
    train_audio_features = ((combined_audio_features.iloc[:len(train_data)] - audio_feature_means) / audio_feature_stds).fillna(0).values; test_audio_features = ((combined_audio_features.iloc[len(train_data):] - audio_feature_means) / audio_feature_stds).fillna(0).values
    fold_feature_matrices = []
    for fold_index, (train_indices, validation_indices) in enumerate(folds):
        fold_train_features, fold_validation_features, fold_test_features = train_features.iloc[train_indices].copy(), train_features.iloc[validation_indices].copy(), test_features.copy()
        inner_target_encodings = compute_out_of_fold_target_encodings(release_dates[train_indices], train_labels[train_indices], track_uris[train_indices], seed * 10 + fold_index)
        fold_train_features["release_date_target_mean"], fold_train_features["release_date_target_count"] = inner_target_encodings[:, 0], inner_target_encodings[:, 1]
        validation_target_encodings = compute_smoothed_target_encodings(release_dates[train_indices], train_labels[train_indices], release_dates[validation_indices]); fold_validation_features["release_date_target_mean"], fold_validation_features["release_date_target_count"] = validation_target_encodings[:, 0], validation_target_encodings[:, 1]
        test_target_encodings = compute_smoothed_target_encodings(release_dates[train_indices], train_labels[train_indices], test_release_dates); fold_test_features["release_date_target_mean"], fold_test_features["release_date_target_count"] = test_target_encodings[:, 0], test_target_encodings[:, 1]
        fold_train_features["same_date_neighbor_label_mean"], fold_train_features["same_date_nearest_neighbor_distance"] = compute_same_date_neighbor_features(train_audio_features[train_indices], release_dates[train_indices], train_labels[train_indices], train_audio_features[train_indices], release_dates[train_indices], exclude_self=True)
        fold_validation_features["same_date_neighbor_label_mean"], fold_validation_features["same_date_nearest_neighbor_distance"] = compute_same_date_neighbor_features(train_audio_features[train_indices], release_dates[train_indices], train_labels[train_indices], train_audio_features[validation_indices], release_dates[validation_indices])
        fold_test_features["same_date_neighbor_label_mean"], fold_test_features["same_date_nearest_neighbor_distance"] = compute_same_date_neighbor_features(train_audio_features[train_indices], release_dates[train_indices], train_labels[train_indices], test_audio_features, test_release_dates)
        fold_feature_matrices.append((fold_train_features, fold_validation_features, fold_test_features))
    return fold_feature_matrices

In [11]:
train_features, test_features = build_static_features(train_data, test_data)
print("Static feature shapes:", train_features.shape, test_features.shape)

Static feature shapes: (3863, 36) (863, 36)


## 3. Validation

In [12]:
duplicate_row_mask = train_features['track_occurrence_count'].to_numpy() > 1
test_duplicate_share = float((test_features['track_occurrence_count'] > 1).mean())
train_duplicate_share = float(duplicate_row_mask.mean())
evaluation_weights = np.where(duplicate_row_mask, test_duplicate_share / (1-test_duplicate_share) * (~duplicate_row_mask).sum() / duplicate_row_mask.sum(), 1.0)
print(f'Duplicate rows: train {train_duplicate_share:.2%}; test {test_duplicate_share:.2%}')


Duplicate rows: train 10.98%; test 2.55%


## 4. Models


In [13]:
LGB_TUNED = {'n_estimators': 1400, 'learning_rate': 0.010425322753516588, 'num_leaves': 22, 'max_depth': 6, 'min_child_samples': 11, 'subsample': 0.5552023046195264, 'colsample_bytree': 0.5720938881088363, 'reg_alpha': 0.001588888980185433, 'reg_lambda': 0.0029158775749992126, 'min_split_gain': 0.024453802531449877, 'extra_trees': True, 'subsample_freq': 1}

XGB_TUNED = {'n_estimators': 800, 'learning_rate': 0.009001416044940976, 'max_depth': 8, 'min_child_weight': 2.455297752945145, 'subsample': 0.6444839011143386, 'colsample_bytree': 0.42155173880638486, 'colsample_bynode': 0.8416675283601944, 'reg_alpha': 0.4578655631243604, 'reg_lambda': 0.2705944540049714, 'gamma': 0.7866098305418372}

CAT_FINAL = {'iterations': 800, 'learning_rate': 0.03, 'depth': 6, 'l2_leaf_reg': 5, 'random_strength': 1, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.5, 'border_count': 128}

EXTRA_TREES_FINAL = {'n_estimators': 800, 'min_samples_leaf': 3, 'max_features': 0.5}

In [14]:
def create_lightgbm_trainer(model_parameters):
    import lightgbm as lgb
    def train_and_predict(fold_train_features, fold_train_labels, fold_validation_features, test_features, seed):
        model = lgb.LGBMClassifier(random_state=seed, verbose=-1, n_jobs=-1, **model_parameters).fit(fold_train_features, fold_train_labels)
        return model.predict_proba(fold_validation_features)[:, 1], model.predict_proba(test_features)[:, 1], model
    return train_and_predict

def create_xgboost_trainer(model_parameters):
    from xgboost import XGBClassifier
    def train_and_predict(fold_train_features, fold_train_labels, fold_validation_features, test_features, seed):
        model = XGBClassifier(random_state=seed, n_jobs=-1, tree_method="hist", **model_parameters).fit(fold_train_features, fold_train_labels)
        return model.predict_proba(fold_validation_features)[:, 1], model.predict_proba(test_features)[:, 1], model
    return train_and_predict

def create_catboost_trainer(model_parameters, task_type="CPU"):
    from catboost import CatBoostClassifier
    def train_and_predict(fold_train_features, fold_train_labels, fold_validation_features, test_features, seed):
        fold_train_features, fold_validation_features, test_features = fold_train_features.copy(), fold_validation_features.copy(), test_features.copy()
        for category_column in CATEGORICAL_FEATURES:
            for feature_frame in (fold_train_features, fold_validation_features, test_features):
                feature_frame[category_column] = feature_frame[category_column].fillna(-1).astype(int).astype(str)
        model = CatBoostClassifier(random_seed=seed, verbose=False, allow_writing_files=False, task_type=task_type,
                               loss_function="Logloss", **model_parameters)
        model.fit(fold_train_features, fold_train_labels, cat_features=CATEGORICAL_FEATURES)
        return model.predict_proba(fold_validation_features)[:, 1], model.predict_proba(test_features)[:, 1], model
    return train_and_predict

def create_extra_trees_trainer(model_parameters):
    from sklearn.ensemble import ExtraTreesClassifier
    from sklearn.impute import SimpleImputer
    def train_and_predict(fold_train_features, fold_train_labels, fold_validation_features, test_features, seed):
        median_imputer = SimpleImputer(strategy="median").fit(fold_train_features)
        model = ExtraTreesClassifier(random_state=seed, n_jobs=-1, **model_parameters).fit(median_imputer.transform(fold_train_features), fold_train_labels)
        return model.predict_proba(median_imputer.transform(fold_validation_features))[:, 1], model.predict_proba(median_imputer.transform(test_features))[:, 1], model
    return train_and_predict

def create_model_trainers(task_type):
    return {
        "lightgbm_tuned": create_lightgbm_trainer(LGB_TUNED),
        "xgboost_tuned": create_xgboost_trainer(XGB_TUNED),
        "catboost_final": create_catboost_trainer(CAT_FINAL, task_type),
        "extra_trees": create_extra_trees_trainer(EXTRA_TREES_FINAL),
    }

def convert_predictions_to_percentile_ranks(prediction_values):
    return rankdata(prediction_values) / len(prediction_values)

In [ ]:
models = {model_name: model_trainer for model_name, model_trainer in create_model_trainers(TASK_TYPE).items() if model_name in MODEL_NAMES}
oof_predictions = {model: np.zeros((len(SEEDS), len(train_labels))) for model in models}
test_predictions = {model: np.zeros((len(SEEDS), len(test_data))) for model in models}
for seed_index, seed in enumerate(SEEDS):
    folds = list(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed).split(train_features, train_labels, groups=train_data["uri"]))
    fold_feature_matrices = build_fold_feature_matrices(train_features, test_features, train_data, test_data, train_labels, folds, seed)
    for model_name, train_and_predict in models.items():
        start_time = time.time()
        for fold_index, (train_indices, validation_indices) in enumerate(folds):
            assert not set(train_data['uri'].iloc[train_indices]) & set(train_data['uri'].iloc[validation_indices])
            fold_train_features, fold_validation_features, fold_test_features = fold_feature_matrices[fold_index]
            validation_probabilities, test_probabilities, model = train_and_predict(fold_train_features, train_labels[train_indices], fold_validation_features, fold_test_features, seed * 100 + fold_index)
            oof_predictions[model_name][seed_index, validation_indices] = validation_probabilities
            test_predictions[model_name][seed_index] += test_probabilities / len(folds)
        oof_auc = roc_auc_score(train_labels, oof_predictions[model_name][seed_index])
        adjusted_auc = roc_auc_score(train_labels, oof_predictions[model_name][seed_index], sample_weight=evaluation_weights)
        print(f"seed {seed} {model_name:16s} OOF AUC={oof_auc:.4f} adjusted={adjusted_auc:.4f} [{time.time()-start_time:.0f}s]", flush=True)

In [ ]:
model_validation_scores = pd.DataFrame({
    model_name: {
        "Out-of-fold ROC AUC": np.mean([
            roc_auc_score(train_labels, predictions)
            for predictions in oof_predictions[model_name]
        ]),
        "Adjusted OOF ROC AUC": np.mean([
            roc_auc_score(train_labels, predictions, sample_weight=evaluation_weights)
            for predictions in oof_predictions[model_name]
        ]),
    }
    for model_name in models
}).T
print("Model validation scores (mean across seeds):")
display(model_validation_scores.style.format('{:.4f}'))

## 5. Ensembling

In [ ]:
model_names = list(models)
ranked_oof_predictions = {
    model_name: np.array([
        convert_predictions_to_percentile_ranks(oof_predictions[model_name][seed_index])
        for seed_index in range(len(SEEDS))
    ])
    for model_name in model_names
}

def calculate_mean_auc(predictions, sample_weights=None):
    return float(np.mean([
        roc_auc_score(train_labels, predictions[seed_index], sample_weight=sample_weights)
        for seed_index in range(len(SEEDS))
    ]))

blend_oof_predictions = np.mean(list(ranked_oof_predictions.values()), axis=0)
blend_weights = {model_name: 1 / len(model_names) for model_name in model_names}
print("Equal rank weights:", blend_weights)
display(pd.DataFrame({
    "Final ensemble": [
        calculate_mean_auc(blend_oof_predictions),
        calculate_mean_auc(blend_oof_predictions, evaluation_weights),
    ]
}, index=["OOF AUC", "Adjusted OOF AUC"]).style.format('{:.4f}'))

In [ ]:
figure, axis = plt.subplots(figsize=(8, 6))
for model_name in model_names:
    model_predictions = oof_predictions[model_name].mean(0)
    false_positive_rates, true_positive_rates, _ = roc_curve(train_labels, model_predictions)
    axis.plot(false_positive_rates, true_positive_rates, alpha=.65, label=f'{model_name} ({roc_auc_score(train_labels, model_predictions):.3f})')
false_positive_rates, true_positive_rates, _ = roc_curve(train_labels, blend_oof_predictions.mean(0))
axis.plot(false_positive_rates, true_positive_rates, color='black', lw=2.5, label=f'blend ({roc_auc_score(train_labels, blend_oof_predictions.mean(0)):.3f})')
axis.plot([0,1], [0,1], '--', color='gray')
axis.set(xlabel='False positive rate', ylabel='True positive rate', title=f'Grouped {N_FOLDS}-fold OOF ROC — seed-averaged predictions')
axis.legend(fontsize=9)
figure.tight_layout(); plt.show(); plt.close(figure)

## 6. Validation scores and submission

In [ ]:
print(f"Blend out-of-fold ROC AUC: {calculate_mean_auc(blend_oof_predictions):.4f}")
print(f"Blend adjusted OOF AUC: {calculate_mean_auc(blend_oof_predictions, evaluation_weights):.4f}")

ranked_test_predictions = {
    model_name: convert_predictions_to_percentile_ranks(test_predictions[model_name].mean(0))
    for model_name in model_names
}
blended_test_predictions = np.mean(list(ranked_test_predictions.values()), axis=0)
submission = sample_submission.copy()
submission[TARGET] = submission['ID'].map(pd.Series(blended_test_predictions, index=test_data['ID']))
submission.to_csv('submission_blend.csv', index=False)
display(submission.head())
files.download('submission_blend.csv')

## 7. Report figures

In [ ]:
release_years = build_release_date_features(train_data)['release_year']
year_bins = [1900, 1980, 1990, 2000, 2010, 2015, 2018, 2020, 2022, 2023, 2024, 2025]
year_labels = ['<1980', '80s', '90s', '00s', '10-14', '15-17', '18-19', '20-21', '2022', '2023', '2024']
year_groups = pd.DataFrame({
    'label': train_labels,
    'era': pd.cut(release_years, year_bins, labels=year_labels, right=False),
}).groupby('era', observed=True)['label'].agg(['mean', 'count'])

figure, axis = plt.subplots(figsize=(9, 4))
axis.bar(year_groups.index.astype(str), year_groups['mean'], color=BLUE, width=.62)
for index, (rate, count) in enumerate(zip(year_groups['mean'], year_groups['count'])):
    axis.text(index, rate + .02, f'n={count}', ha='center', fontsize=8, color=MUTED)
axis.axhline(train_labels.mean(), color=MUTED, ls='--', lw=1)
axis.set(xlabel='Album release year', ylabel='Share of High-popularity tracks', ylim=(0, 1.08), title='Release era strongly separates popularity labels')
axis.grid(axis='x', visible=False)
figure.tight_layout()
plt.show()

In [ ]:
uri_counts = train_data.groupby('uri')['uri'].transform('count')
count_groups = pd.DataFrame({'label': train_labels, 'count': uri_counts}).groupby('count')['label'].agg(['mean', 'count'])
unique_tracks = train_data.drop_duplicates('uri').reset_index(drop=True)
unique_labels = unique_tracks['high_popularity_label'].to_numpy()
neighbor_audio = unique_tracks[KNN_AUDIO_FEATURES].copy()
neighbor_audio['duration_ms'] = np.log1p(neighbor_audio['duration_ms'])
neighbor_audio['instrumentalness'] = np.log10(neighbor_audio['instrumentalness'] + 1e-6)
neighbor_audio = ((neighbor_audio - neighbor_audio.mean()) / neighbor_audio.std()).fillna(0).to_numpy()
release_dates = unique_tracks['track_album_release_date'].astype(str).to_numpy()
release_year = pd.to_numeric(pd.Series(release_dates).str[:4], errors='coerce').to_numpy()
indices_by_date = {}
for index, release_date in enumerate(release_dates):
    indices_by_date.setdefault(release_date, []).append(index)

random_generator = np.random.default_rng(0)
random_agreement, same_year_agreement, nearest_date_agreement = [], [], []
for index, release_date in enumerate(release_dates):
    candidates = np.array([candidate for candidate in indices_by_date[release_date] if candidate != index])
    if len(candidates) == 0:
        continue
    distances = np.sqrt(((neighbor_audio[candidates] - neighbor_audio[index]) ** 2).sum(1))
    nearest = candidates[np.argmin(distances)]
    same_year = np.where(release_year == release_year[index])[0]
    same_year = same_year[same_year != index]
    nearest_date_agreement.append(unique_labels[nearest] == unique_labels[index])
    same_year_agreement.append(unique_labels[random_generator.choice(same_year)] == unique_labels[index])
    random_agreement.append(unique_labels[random_generator.integers(len(unique_labels))] == unique_labels[index])

figure, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar([str(value) for value in count_groups.index], count_groups['mean'], color=BLUE, width=.55)
for index, (rate, count) in enumerate(zip(count_groups['mean'], count_groups['count'])):
    axes[0].text(index, rate + .02, f'{rate:.0%}\n(n={count})', ha='center', fontsize=8)
axes[0].set(xlabel='Times track appears in train.csv', ylabel='Share High', ylim=(0, 1.2), title='Track multiplicity')
axes[0].grid(axis='x', visible=False)
agreement_values = [np.mean(random_agreement), np.mean(same_year_agreement), np.mean(nearest_date_agreement)]
axes[1].barh(['Random track', 'Random same-year track', 'Nearest same-date track'], agreement_values, color=[MUTED, MUTED, ORANGE])
for index, value in enumerate(agreement_values):
    axes[1].text(value + .01, index, f'{value:.2f}', va='center', fontsize=9)
axes[1].set(xlim=(.5, .8), xlabel='Label agreement', title='Album-level signal')
axes[1].grid(axis='y', visible=False)
figure.tight_layout()
plt.show()

In [ ]:
progression = pd.DataFrame({
    'Experiment': ['Audio only', '+ release date', '+ URI count', '+ album and KNN', '+ Optuna tuning', 'Final ensemble'],
    'OOF AUC': [0.7517, 0.8325, 0.8497, 0.8536, 0.8571, 0.8646],
    'Adjusted OOF AUC': [np.nan, 0.8292, 0.8353, 0.8375, 0.8410, 0.8487],
})
comparison_scores = model_validation_scores['Adjusted OOF ROC AUC'].copy()
comparison_scores.loc['Final ensemble'] = calculate_mean_auc(blend_oof_predictions, evaluation_weights)
comparison_scores = comparison_scores.sort_values()

figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
y_positions = np.arange(len(progression))
axes[0].barh(y_positions + .18, progression['OOF AUC'], height=.34, color=BLUE, label='OOF AUC')
axes[0].barh(y_positions - .18, progression['Adjusted OOF AUC'], height=.34, color=ORANGE, label='Adjusted OOF AUC')
axes[0].set_yticks(y_positions, progression['Experiment'])
axes[0].set(xlim=(.70, .88), xlabel='ROC AUC', title='Validation progression')
axes[0].legend(frameon=False, fontsize=8)
axes[0].grid(axis='y', visible=False)
bar_colors = [ORANGE if name == 'Final ensemble' else BLUE for name in comparison_scores.index]
axes[1].barh(comparison_scores.index, comparison_scores.values, color=bar_colors)
for index, value in enumerate(comparison_scores.values):
    axes[1].text(value + .0003, index, f'{value:.4f}', va='center', fontsize=8)
axes[1].set(xlim=(comparison_scores.min() - .004, comparison_scores.max() + .004), xlabel='Adjusted OOF AUC', title='Final model comparison')
axes[1].grid(axis='y', visible=False)
figure.tight_layout()
plt.show()